In [ ]:
dev_id = "20260723-032552-e46d2b"
val_id = "20260723-032553-e79bce"

In [ ]:
import importlib, ddi.data, ddi.mask, ddi.manifest, ddi.train, ddi.run
for m in (ddi.data, ddi.manifest, ddi.mask, ddi.train, ddi.run):
    importlib.reload(m)

from ddi.data import build_human
from ddi.manifest import write_dataset, load_dataset
from ddi.mask import derive_masked, derive_masked_pair
from transformers import AutoTokenizer

MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
SYNTH_ID = "20260727-121745-bafaa8"

tok = AutoTokenizer.from_pretrained(MODEL)
print(tok.tokenize("drug1 interacts with drug2 and drug0"))

    
train, dev, val = build_human()
train_id = write_dataset(train, provenance="human:train", seed=42,
                         notes="full human train, markers")

tr_mark_id, tr_mask_id, tr_dropped = derive_masked_pair(train_id)
dev_mark_id, dev_mask_id, dev_dropped = derive_masked_pair(dev_id)
sy_mark_id, sy_mask_id, sy_dropped = derive_masked_pair(SYNTH_ID)

for r in dev_dropped[:5]:
    print(r["label"], r["text"][:200])

print(tok.tokenize("drug1 interacts with drug2 and drug0"))

In [ ]:
from ddi.experiment import load_runs
df = load_runs()
df[df["m.micro_f1_pos"].between(0.28, 0.29)][["run_id", "train_id", "m.micro_f1_pos"]]

In [ ]:
import pandas as pd
from ddi.run import run_training

BASE = {"model_name": MODEL, "epochs": 3, "lr": 2e-5,
        "batch_size": 32, "max_length": 256, "neg_ratio": None}

ARMS = [
    ("human",     "markers",      "unaligned", train_id,     dev_id),
    ("human",     "markers",      "aligned",   tr_mark_id,   dev_mark_id),
    ("human",     "mask_targets", "aligned",   tr_mask_id,   dev_mask_id),
    ("synthetic", "markers",      "aligned",   sy_mark_id,   dev_mark_id),
    ("synthetic", "mask_targets", "aligned",   sy_mask_id,   dev_mask_id),
]

rows = []
for dataset, mode, align, tr_id, ev_id in ARMS:
    for seed in [0, 1, 2]:
        cfg = {**BASE, "seed": seed, "dataset": dataset, "render_mode": mode}
        _, m = run_training(tr_id, ev_id, cfg, notes=f"masking {dataset}/{mode}/{align}")
        rows.append({"dataset": dataset, "mode": mode, "align": align, "seed": seed,
                     "f1": m["micro_f1_pos"], "p": m["micro_p_pos"], "r": m["micro_r_pos"],
                     "f1_DrugBank": m.get("micro_f1_pos_DrugBank"),
                     "f1_MedLine": m.get("micro_f1_pos_MedLine")})
        print(f"{dataset:10s} {mode:13s} {align:9s} seed={seed}  f1={m['micro_f1_pos']:.3f}")

df = pd.DataFrame(rows)
df.groupby(["dataset", "mode", "align"])[["f1", "p", "r", "f1_DrugBank", "f1_MedLine"]].agg(["mean", "std"])